In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import optuna
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.saakyan.lesson3 import Exercise

In [ ]:
X, y = load_digits(return_X_y=True)

X = X.astype(np.float32) / 16.0
y = y.astype(np.int64)

In [ ]:
print(X.shape, y.shape)

In [ ]:
X[0], y[1]

In [ ]:
i = 0
plt.matshow(X[i].reshape(8, 8), cmap="grey")
print(y[i])
plt.show()

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

In [ ]:
STORAGE = "sqlite:///students/saakyan/optuna_digits.db"
STUDY_NAME = "digits_saakyan"


def build_model(hidden_dim):
    return Exercise.create_model(
        Exercise.create_linear_layer(64, hidden_dim),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_dim, 64),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(64, 10),
        Exercise.create_logsoftmax_layer(),
    )

In [ ]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    n_epoch = trial.suggest_int("n_epoch", 20, 100)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256])

    model = build_model(hidden_dim)
    loss_fn = Exercise.create_nll_loss()

    Exercise.train_model(model, loss_fn, X_train, y_train, lr, n_epoch, batch_size)

    val_log_probs = model.forward(X_val)
    val_preds = np.argmax(val_log_probs, axis=1)
    val_acc = np.mean(val_preds == y_val)

    return float(val_acc)


study = optuna.create_study(study_name=STUDY_NAME, storage=STORAGE, direction="maximize", load_if_exists=True)
study.optimize(objective, n_trials=50)

print("Лучшие параметры:", study.best_params)
print("Лучшая val accuracy:", study.best_value)

In [ ]:
best_params = study.best_params

best_model = build_model(best_params["hidden_dim"])
best_loss_fn = Exercise.create_nll_loss()

Exercise.train_model(
    best_model,
    best_loss_fn,
    X_train_val,
    y_train_val,
    best_params["lr"],
    best_params["n_epoch"],
    best_params["batch_size"],
)

In [ ]:
test_log_probs = best_model.forward(X_test)
test_preds = np.argmax(test_log_probs, axis=1)
test_loss = -np.mean(test_log_probs[np.arange(len(y_test)), y_test])
test_acc = np.mean(test_preds == y_test)

print("test_loss =", round(float(test_loss), 4))
print("test_acc =", round(float(test_acc), 4))
print("best_params =", best_params)